In [11]:
%run ./02_build_sparksession_energy_generation.ipynb
clear_output(wait=True)

Load raw dataset from parquet files.

In [13]:
#Read rawdata delta tables to cleand, validate and generate new ones
# 1. Leer de la base de datos anterior
df_bronze_energy_generation = spark_energy.read.format("iceberg").load("local.db_raw.energy_generations")

df_bronze_energy_generation.show(5)

+-----+----------+----+---------+------------------+--------------------+----------------+---------------------------+-------------------------+-----------------------+-----------------------------------------+------------------+-----------------------------+------------------------+----------------------+-------------------------------+---------+------------+---------+-----------+----------+---------+------------+
| Area|ISO 3 code|Year|Area type|Electricity source|Is aggregated source|Generation (TWh)|Generation YoY change (TWh)|Generation YoY change (%)|Share of generation (%)|Share of generation YoY change (% points)|Emissions (MtCO2e)|Emissions YoY change (MtCO2e)|Emissions YoY change (%)|Share of emissions (%)|Emissions intensity (gCO2e/kWh)|Continent|Ember region|EU member|OECD member|G20 member|G7 member|ASEAN member|
+-----+----------+----+---------+------------------+--------------------+----------------+---------------------------+-------------------------+------------------

Rename columns to keep them standarzed for business analytics

In [16]:
# 2. Renombrar las columnas de dos palabras usando .withColumnRenamed()
df_silver_energy_transformation = df_bronze_energy_generation \
   .withColumnRenamed("Area", "area") \
   .withColumnRenamed("ISO 3 code", "iso_3_code") \
   .withColumnRenamed("Year", "year") \
   .withColumnRenamed("Area type", "area_type") \
   .withColumnRenamed("Electricity_source", "electricity_source") \
   .withColumnRenamed("Is Aggregated source", "is_aggregated_source") \
   .withColumnRenamed("Generation (TWh", "generation_twh") \
   .withColumnRenamed("Generation YoY change (TWh)", "generation_yoy_change_twh") \
   .withColumnRenamed("Generation YoY change (%)", "generation_yoy_change_%") \
   .withColumnRenamed("Share of generation (%)", "share_of_generation_%") \
   .withColumnRenamed("Emissions (MtCO2e)", "emissions_mtco2e") \
   .withColumnRenamed("Emissions YoY change (MtCO2e)", "emissions yoy change mtco2e") \
   .withColumnRenamed("Share of emissions (%)", "share_of_emissions_%") \
   .withColumnRenamed("Emissions intensity (gCO2e/kWh)", "emissions intensity_gco2e_per_kwh") \
   .withColumnRenamed("Continent", "Continent") \
   .withColumnRenamed("Ember region", "ember_region") \
   .withColumnRenamed("EU member", "eu_member") \
   .withColumnRenamed("OECD member", "oecd_member") \
   .withColumnRenamed("G20 member", "g20_member") \
   .withColumnRenamed("G7 member", "g7_member") \
   .withColumnRenamed("ASEAN member", "asean_member") \
#    .fillna({"generation_yoy_change_twh": 0.0, "region": "unknown"}) \
#    .dropna(subset=["energy_generation"]) \
#    .dropDuplicates()

df_silver_energy_transformation.show(2)


+-----+----------+----+---------+------------------+--------------------+----------------+-------------------------+-----------------------+---------------------+-----------------------------------------+----------------+---------------------------+------------------------+--------------------+---------------------------------+---------+------------+---------+-----------+----------+---------+------------+
| area|iso_3_code|year|area_type|Electricity source|is_aggregated_source|Generation (TWh)|generation_yoy_change_twh|generation_yoy_change_%|share_of_generation_%|Share of generation YoY change (% points)|emissions_mtco2e|emissions yoy change mtco2e|Emissions YoY change (%)|share_of_emissions_%|emissions intensity_gco2e_per_kwh|Continent|ember_region|eu_member|oecd_member|g20_member|g7_member|asean_member|
+-----+----------+----+---------+------------------+--------------------+----------------+-------------------------+-----------------------+---------------------+--------------------

Impute Missing Values

In [2]:
#create respository for silver layer
spark_energy.sql("CREATE DATABASE IF NOT EXISTS db_silver.energy_generations")

In [1]:
# create delta table me
df_bronze_energy_generation.writeTo("local.db_silver.energy_generations") \
    .tableProperty("format-version", "2") \
    .createOrReplace()

print("✅ ¡Tabla creada exitosamente en `data_lakehouse/db_silvert.db/energy_generations`!"

In [5]:
!ls ./

01_eda_generation_energy.ipynb
02_build_sparksession_energy_generation.ipynb
03_rawdata_global_energy_generation.ipynb
04_silver_transformation_energy_generation.ipynb
05_gold_metrics_energy_generation.ipynb
ProcessDataWithSparkIceberg.ipynb
